In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/project")


def read_env(path):
    values = {}
    for number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key or key != key.strip():
            raise ValueError(f"Invalid KEY=value line {number} in {path}")
        values[key] = value
    return values


CONFIG = read_env(PROJECT_DIR / ".colab.env")
R2_CREDENTIALS = read_env(Path("/content/.colab-r2.env"))
for key in ("R2_BUCKET", "R2_ARTIFACT_PREFIX", "EXPECT_GPU"):
    if not CONFIG.get(key):
        raise ValueError(f"Missing {key} in .colab.env")
for key in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    if not R2_CREDENTIALS.get(key):
        raise ValueError(f"Missing {key} in the external R2 credentials file")
if CONFIG["EXPECT_GPU"] not in ("true", "false"):
    raise ValueError("EXPECT_GPU must be true or false")
ARTIFACT_PREFIX = CONFIG["R2_ARTIFACT_PREFIX"].strip("/")
if not ARTIFACT_PREFIX:
    raise ValueError("R2_ARTIFACT_PREFIX must name a folder")
if CONFIG.get("R2_DATA_PREFIX", "").strip("/") != ARTIFACT_PREFIX:
    raise ValueError("R2_DATA_PREFIX must match R2_ARTIFACT_PREFIX")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

gpu_command = shutil.which("nvidia-smi")
gpu = (
    subprocess.run([gpu_command, "-L"], capture_output=True, text=True)
    if gpu_command
    else None
)
if CONFIG["EXPECT_GPU"] == "true" and (gpu is None or gpu.returncode != 0):
    raise RuntimeError("GPU expected but nvidia-smi did not find one")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"Working directory: {Path.cwd()}")
print(f"GPU: {gpu.stdout.strip() if gpu and gpu.returncode == 0 else 'none'}")
print(f"R2 bucket: {CONFIG['R2_BUCKET']} | data: {DATA_DIR} | output: {OUTPUT_DIR}")
print(f"Artifact prefix: {ARTIFACT_PREFIX}")
print("R2 credentials: present")

In [ ]:
%pip install -qq --disable-pip-version-check uv boto3

import subprocess

requirements = PROJECT_DIR / "requirements-colab.txt"
subprocess.run(
    [
        "uv",
        "export",
        "--frozen",
        "--no-dev",
        "--extra",
        "experiment",
        "--prune",
        "torch",
        "--prune",
        "numpy",
        "--prune",
        "fsspec",
        "--prune",
        "rich",
        "--prune",
        "colorama",
        "--no-emit-project",
        "--no-hashes",
        "--format",
        "requirements.txt",
        "--output-file",
        str(requirements),
        "--project",
        str(PROJECT_DIR),
    ],
    check=True,
)
%pip install -qq --disable-pip-version-check --requirement {requirements}
%pip install -qq --disable-pip-version-check --no-deps {PROJECT_DIR}

In [ ]:
from jlens_reasoning.environments.colab import source_bundle_sha256

PROJECT_SOURCE_SHA256 = source_bundle_sha256(PROJECT_DIR)
print(f"Project source SHA-256: {PROJECT_SOURCE_SHA256}")

In [ ]:
def r2_client():
    import boto3

    return boto3.client(
        "s3",
        endpoint_url=f"https://{R2_CREDENTIALS['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_CREDENTIALS["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=R2_CREDENTIALS["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )


def upload_artifacts():
    """Upload files in OUTPUT_DIR under the project's artifact prefix."""
    client = r2_client()
    count = 0
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_symlink():
            raise ValueError(f"Refusing to upload symlink: {path}")
        if path.is_file():
            key = f"{ARTIFACT_PREFIX}/{path.relative_to(OUTPUT_DIR).as_posix()}"
            client.upload_file(str(path), CONFIG["R2_BUCKET"], key)
            count += 1
    print(f"Uploaded {count} file(s) from {OUTPUT_DIR}")

In [ ]:
from jlens_reasoning.environments.colab import download_r2_inputs

download_r2_inputs(
    client=r2_client(),
    bucket=CONFIG["R2_BUCKET"],
    prefix=ARTIFACT_PREFIX,
    destination=DATA_DIR,
    paths=(
        "assets/models/qwen3.5-4b/",
        "assets/lenses/qwen3.5-4b/",
        "datasets/flenqa/",
    ),
)

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from collections import Counter

import jlens
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import transformers
from datasets import load_from_disk
from tqdm.auto import tqdm

from experiments.jlens_readout_sanity.constants import (
    LENS_PATH,
    MODEL_NAME,
    MODEL_PATH,
)
from jlens_reasoning.benchmarks.flenqa.dataset import (
    normalize_rows,
    prepare_prompts,
)
from jlens_reasoning.benchmarks.flenqa.lens import (
    ApplyLensRunner,
    LensRunners,
)
from jlens_reasoning.benchmarks.flenqa.runner import (
    RunConfig,
    run_benchmark,
)
from jlens_reasoning.inference import InferenceConfig, generate_chat

dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
runners = LensRunners(
    ApplyLensRunner(lens, model, True), ApplyLensRunner(lens, model, False)
)

In [ ]:
len(rows)

In [ ]:
summary = run_benchmark(
    rows,
    output_dir=OUTPUT_DIR / "runs/flenqa-full-run",
    tokenizer=tokenizer,
    runners=runners,
    config=RunConfig(
        top_k=250,
        expected_source_rows=12_000,
    ),
    overwrite=True,
)
summary

In [ ]:
# Rerunning this cell overwrites model_outputs.parquet with newly generated answers.
# For answer regeneration, run setup/loading cells and skip run-benchmark.
MODEL_OUTPUT_PATH = OUTPUT_DIR / "runs/flenqa-full-run" / "model_outputs.parquet"
LENGTHS = (250, 500, 1000, 2000, 3000)
EXPECTED_UNIQUE_COUNTS = {
    250: 300,
    500: 2_368,
    1000: 2_394,
    2000: 2_400,
    3000: 2_400,
}
INFERENCE_CONFIG = InferenceConfig.direct(max_input_tokens=4096)
MODEL_OUTPUT_SCHEMA = pa.schema(
    [
        pa.field("prompt_id", pa.string(), nullable=False),
        pa.field("problem_id", pa.int32(), nullable=False),
        pa.field("task", pa.string(), nullable=False),
        pa.field("label", pa.bool_(), nullable=False),
        pa.field("text", pa.string(), nullable=False),
        pa.field("ctx_size", pa.int32(), nullable=False),
        pa.field("n_input_tokens", pa.int32(), nullable=False),
        pa.field("input_sha256", pa.string(), nullable=False),
        pa.field("paper_weight", pa.int16(), nullable=False),
        pa.field("model_name", pa.string(), nullable=False),
        pa.field("code_revision", pa.string(), nullable=False),
        pa.field("inference_mode", pa.string(), nullable=False),
        pa.field("max_new_tokens", pa.int32(), nullable=False),
        pa.field("do_sample", pa.bool_(), nullable=False),
        pa.field("temperature", pa.float32()),
        pa.field("top_p", pa.float32()),
        pa.field("top_k", pa.int32()),
        pa.field("min_p", pa.float32()),
        pa.field("generated_token_ids", pa.list_(pa.int32()), nullable=False),
        pa.field("generated_token_pieces", pa.list_(pa.string()), nullable=False),
        pa.field("generated_text", pa.string(), nullable=False),
        pa.field("reasoning_text", pa.string()),
        pa.field("answer_text", pa.string()),
        pa.field("reasoning_status", pa.string(), nullable=False),
        pa.field("generation_status", pa.string(), nullable=False),
        pa.field("finish_reason", pa.string()),
    ]
)

prompts = prepare_prompts(rows)
records = []
for prompt in tqdm(prompts, desc="FLenQA model outputs", unit="prompt"):
    ctx_sizes = {item.ctx_size for item in prompt.provenance}
    if len(ctx_sizes) != 1:
        raise ValueError(f"Prompt spans nominal lengths: {sorted(ctx_sizes)}")
    inference = generate_chat(
        causal_lm,
        tokenizer,
        prompt.text,
        config=INFERENCE_CONFIG,
    )
    records.append(
        {
            "prompt_id": prompt.prompt_id,
            "problem_id": prompt.problem_id,
            "task": prompt.task,
            "label": prompt.label,
            "text": prompt.text,
            "ctx_size": ctx_sizes.pop(),
            "n_input_tokens": inference.input_token_count,
            "input_sha256": inference.input_sha256,
            "paper_weight": sum(
                item.dispersion == "random" for item in prompt.provenance
            ),
            "model_name": MODEL_NAME,
            "code_revision": PROJECT_SOURCE_SHA256,
            "inference_mode": inference.config.mode.value,
            "max_new_tokens": inference.config.max_new_tokens,
            "do_sample": inference.config.do_sample,
            "temperature": inference.config.temperature,
            "top_p": inference.config.top_p,
            "top_k": inference.config.top_k,
            "min_p": inference.config.min_p,
            "generated_token_ids": list(inference.output.token_ids),
            "generated_token_pieces": list(inference.output.token_pieces),
            "generated_text": inference.raw_text,
            "reasoning_text": inference.reasoning_text,
            "answer_text": inference.answer_text,
            "reasoning_status": inference.reasoning_status.value,
            "generation_status": inference.output.generation_status.value,
            "finish_reason": inference.output.finish_reason,
        }
    )

assert len(records) == 9_862
actual_counts = Counter(record["ctx_size"] for record in records)
assert dict(actual_counts) == EXPECTED_UNIQUE_COUNTS
paper_counts = Counter()
for record in records:
    paper_counts[record["ctx_size"]] += record["paper_weight"]
assert dict(paper_counts) == {length: 600 for length in LENGTHS}

model_outputs = pa.Table.from_pylist(records, schema=MODEL_OUTPUT_SCHEMA)
assert model_outputs.num_rows == 9_862
pq.write_table(model_outputs, MODEL_OUTPUT_PATH, compression="zstd")
MODEL_OUTPUT_PATH

In [ ]:
upload_artifacts()
print("COLAB_NOTEBOOK_UPLOAD_COMPLETE")